In [10]:
# Algebraic promotion
import numpy as np
import stim
from tqdm import tqdm

defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0],
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])
def printnice(shots):
    for i in range(len(shots)):
        print(np.array(shots[i]))

def base4(n):
    digits = [0] * 8
    i = 0
    while n:
        digits[i] = n % 4
        n //= 4
        i += 1
    return digits[::-1]

def toDecimal(binary):
    return int(binary, 2)
def toBinary(decimal):
    return format(decimal, '08b')
def parity(x):
    p = 0
    while x:
        p ^= 1
        x &= x - 1    # removes the lowest set bit
    return p          # 0 = even, 1 = odd
def measure_logical(log, readout):
    count = 0
    for i in range(len(readout)):
        if readout[i] == 1 and log[i] == '1':
            count += 1
    return count % 2
def logical_to_string(log):
    out = ''
    for i in range(len(log)):
        out += 'Z' if log[i] == '1' else 'I'
    return out
def weight(bitstring):
    return sum([int(bit) for bit in bitstring])

In [2]:
# Basic simulator
errortypes = ['I', 'X', 'Y', 'Z']
def measure_schedules(circuit, zschedule, xschedule):
    for row in zschedule:
        stabilizerstring = ''
        for i in range(len(row)):
            if row[i] != 0:
                stabilizerstring += 'Z' + str(i) + '*'
        circuit.append('MPP', stim.PauliString(stabilizerstring[:-1]))
    for row in xschedule:
        stabilizerstring = ''
        for i in range(len(row)):
            if row[i] != 0:
                stabilizerstring += 'X' + str(i) + '*'
        circuit.append('MPP', stim.PauliString(stabilizerstring[:-1]))
    return circuit

def simulateError(zschedule = defaultzschedule, xschedule = defaultxschedule, errors = ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'], nshots = 100000):
    circuit = stim.Circuit()
    circuit.append('R', [0, 1, 2, 3, 4, 5, 6, 7])
    measure_schedules(circuit, zschedule, xschedule)
    for i in range(len(errortypes)):
        qubitlist = []
        for j in range(len(errors)):
            if errors[j] == errortypes[i]:
                qubitlist.append(j)
        circuit.append(errortypes[i], qubitlist)
    measure_schedules(circuit, zschedule, xschedule)
    circuit.append('M', [0, 1, 2, 3, 4, 5, 6, 7])
    sampler = circuit.compile_sampler()
    shots = sampler.sample(nshots)
    return shots.astype(int)

results = simulateError(errors = ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'], nshots = 4)
detector, readout = (results[:, :5] + results[:, 5:10]) % 2, results[:, 10:]
for i in range(len(detector)):
    print(detector[i], readout[i])

[0 0 0 0 0] [1 1 1 1 1 1 1 1]
[0 0 0 0 0] [1 1 1 1 1 1 1 1]
[0 0 0 0 0] [0 0 0 0 0 0 0 0]
[0 0 0 0 0] [1 1 1 1 1 1 1 1]


In [3]:
# Create set of all logical operators
row, col = 0, 4 

logicaloperators = {}
for i in range(1, 256):
    if parity(i) == 0:
        logicaloperators[toBinary(i)] = 0
print(len(logicaloperators), logicaloperators.keys())


127 dict_keys(['00000011', '00000101', '00000110', '00001001', '00001010', '00001100', '00001111', '00010001', '00010010', '00010100', '00010111', '00011000', '00011011', '00011101', '00011110', '00100001', '00100010', '00100100', '00100111', '00101000', '00101011', '00101101', '00101110', '00110000', '00110011', '00110101', '00110110', '00111001', '00111010', '00111100', '00111111', '01000001', '01000010', '01000100', '01000111', '01001000', '01001011', '01001101', '01001110', '01010000', '01010011', '01010101', '01010110', '01011001', '01011010', '01011100', '01011111', '01100000', '01100011', '01100101', '01100110', '01101001', '01101010', '01101100', '01101111', '01110001', '01110010', '01110100', '01110111', '01111000', '01111011', '01111101', '01111110', '10000001', '10000010', '10000100', '10000111', '10001000', '10001011', '10001101', '10001110', '10010000', '10010011', '10010101', '10010110', '10011001', '10011010', '10011100', '10011111', '10100000', '10100011', '10100101', '

In [33]:
#zschedule = defaultzschedule
zschedule = np.load('schedules/schedule_11000000.npy')
zslen = zschedule.shape[0]
print(zslen, zschedule)

5 [[1 2 0 0 0 0 0 0]
 [2 0 1 0 3 0 4 0]
 [0 0 2 1 0 0 0 0]
 [0 0 0 0 1 2 0 0]
 [0 0 0 0 0 0 1 2]]


In [32]:
p = 0.1

logicaloperators = {}
for i in range(1, 256):
    if parity(i) == 0:
        logicaloperators[toBinary(i)] = 0
print(len(logicaloperators), logicaloperators.keys())

ler = 0
conditionaller = 0
it = 5
totalp = (1-3*p)**8
sneakyreadouts = []
for i in tqdm(range(1, 4**8)):
    errors = [errortypes[j] for j in base4(i)]
    perror = 1
    for error in errors:
        if error == 'I':
            perror *= 1 - 3*p
        else:
            perror *= p
    totalp += perror
    result = simulateError(zschedule = zschedule, errors = errors, nshots = 1)
    #print(result)
    detector, readout = (result[:, :zslen + 1] + result[:, zslen+1:2*zslen+2]) % 2, result[:, 2*zslen+2:]
    #print(detector, readout)
    for log in logicaloperators:
        if measure_logical(log, readout[0]) == 1 and np.sum(detector[0][0:4]) == 0:
        #if measure_logical(log, readout[0]) == 1 and np.sum(detector[0:5]) == 0:
        #if measure_logical(log, readout[0]) == 1:
            logicaloperators[log] += perror
            sneakyreadouts.append(readout[0])
sneakyreadouts = np.array(sneakyreadouts)
sneakyreadouts = np.unique(sneakyreadouts, axis=0)
print(len(sneakyreadouts))

print(totalp)

127 dict_keys(['00000011', '00000101', '00000110', '00001001', '00001010', '00001100', '00001111', '00010001', '00010010', '00010100', '00010111', '00011000', '00011011', '00011101', '00011110', '00100001', '00100010', '00100100', '00100111', '00101000', '00101011', '00101101', '00101110', '00110000', '00110011', '00110101', '00110110', '00111001', '00111010', '00111100', '00111111', '01000001', '01000010', '01000100', '01000111', '01001000', '01001011', '01001101', '01001110', '01010000', '01010011', '01010101', '01010110', '01011001', '01011010', '01011100', '01011111', '01100000', '01100011', '01100101', '01100110', '01101001', '01101010', '01101100', '01101111', '01110001', '01110010', '01110100', '01110111', '01111000', '01111011', '01111101', '01111110', '10000001', '10000010', '10000100', '10000111', '10001000', '10001011', '10001101', '10001110', '10010000', '10010011', '10010101', '10010110', '10011001', '10011010', '10011100', '10011111', '10100000', '10100011', '10100101', '

 11%|█         | 7201/65535 [00:03<00:28, 2027.27it/s]


KeyboardInterrupt: 

In [5]:
result = simulateError(errors = ['X', 'I', 'X', 'I', 'X', 'I', 'X', 'Z'], nshots = 1)
detector, readout = (result[:, :5] + result[:, 5:10]) % 2, result[:, 10:]
print(detector, readout)


[[0 0 0 0 1]] [[0 1 0 1 0 1 0 1]]


In [31]:
for log, val in sorted(logicaloperators.items(), key=lambda x: x[1]):
    print(f"{(log)}: {val}")

00001111: 0
00110011: 0
00111100: 0
01010101: 0
01011010: 0
01100110: 0
01101001: 0
10010110: 0
10011001: 0
10100101: 0
10101010: 0
11000011: 0
11001100: 0
11110000: 0
11111111: 0
00000011: 0.0052428799999997
00001100: 0.0052428799999997
00110000: 0.0052428799999997
00111111: 0.0052428799999997
01010110: 0.0052428799999997
01011001: 0.0052428799999997
01100101: 0.0052428799999997
01101010: 0.0052428799999997
10010101: 0.0052428799999997
10011010: 0.0052428799999997
10100110: 0.0052428799999997
10101001: 0.0052428799999997
11000000: 0.0052428799999997
11001111: 0.0052428799999997
11110011: 0.0052428799999997
11111100: 0.0052428799999997
00000101: 0.005242879999999705
00001010: 0.005242879999999705
00110110: 0.005242879999999705
00111001: 0.005242879999999705
01010000: 0.005242879999999705
01011111: 0.005242879999999705
01100011: 0.005242879999999705
01101100: 0.005242879999999705
10010011: 0.005242879999999705
10011100: 0.005242879999999705
10100000: 0.005242879999999705
10101111: 0.005